# 05 · Model Karşılaştırma Eğitimleri

**Amaç:** `configs/experiments.json` içinde tanımlı mimarileri aynı veri bölmesi, aynı sınıf ağırlığı ve aynı erken durdurma kuralıyla eğitip Drive'a kaydetmek.

* Bu defter **sadece eğitim** yapar; karşılaştırma tablosu ve grafikler `06_compare_results.ipynb` defterinde üretilir.
* Her model **bağımsız bir hücredir**; istediğinizi tek tek çalıştırabilirsiniz.
* `results.json` zaten varsa hücre o modeli atlar; eğitim yarıda kalmışsa `last.pt` üzerinden kaldığı turdan devam eder.
* **Test kümesi bu defterde hiç kullanılmaz.** Eşik doğrulama kümesinde seçildiği için buradaki eşikli metrikler (F1, Precision, Recall) hafifçe iyimser olabilir; test aşamasında bu eşik sabit tutulacaktır.

| Hücre | Deney klasörü | Model | Durum | Amaç |
|---|---|---|---|---|
| 1 | `05_cnn_avg` | `TornadoCNN (pool="avg")` | Zorunlu | Global Max vs Global Avg havuzlama ablasyonu |
| 2 | `05_resnet18_pretrained` | `ResNet-18 (ön-eğitimli, lr=1e-4)` | Zorunlu | Transfer öğrenme karşılaştırması |
| 3 | `05_resnet18_scratch` | `ResNet-18 (sıfırdan, lr=1e-4)` | Zorunlu | Ön-eğitimin etkisini aynı `lr` ile ölçme |
| 4 | `05_cnn_bn` | `TornadoCNN (pool="max", batch_norm=True)` | İsteğe bağlı | BatchNorm etkisi |
| 5 | Şablon | Toplu veya çoklu seed koşumu | İsteğe bağlı | Config'deki yeni/ek deneyler |

In [1]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import json, os, sys
from pathlib import Path

BOOTSTRAP_BRANCH = os.environ.get("TORNADO_BRANCH", "main")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri ve modeller Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q -b {branch} https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} fetch -q origin {branch}
        !git -C {REPO} checkout -q {branch}
        !git -C {REPO} pull -q origin {branch}
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH

sys.path.insert(0, str(REPO))
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO, "| dal:", branch)

Mounted at /content/drive
Colab | repo: /content/tornado-cnn | dal: main


In [2]:
import torch
from src.helper import model as cnn
from src.helper import paths, train

cfg_data = paths.load_config("data.json")
cfg = paths.load_config("train.json")
exp_registry = paths.load_config("experiments.json")
EXPERIMENTS = {e["name"]: e for e in exp_registry.get("baselines", []) + exp_registry["experiments"]}

device = train.get_device()
if device.type == "cuda":
    print("cihaz: GPU |", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU yok, eğitim çok yavaş olur. Çalışma zamanı → T4 GPU seçin.")
print("Tanımlı deneyler:", list(EXPERIMENTS))
cfg

cihaz: GPU | Tesla T4
Tanımlı deneyler: ['03_baseline', '03_mit', '05_cnn_avg', '05_resnet18_pretrained', '05_resnet18_scratch', '05_cnn_bn', '05_cnn_coord', '05_cnn_aug', '05_cnn_ls']


{'batch_size': 64,
 'optimizer': 'adam',
 'lr': 0.001,
 'weight_decay': 0.0,
 'epochs': 20,
 'patience': 4,
 'dropout': 0.3,
 'weighting': 'pos_weight',
 'monitor': 'pr_auc',
 'augment': False,
 'label_smoothing': 0.0,
 'seed': 42,
 'num_workers': 2}

In [3]:
HIZLI_DENEME = False     # True: sadece 2017 ile hızlı deneme (karar G1: nihai sonuçlar 3 yılla)

data_root, models_root = paths.prepare_colab_dirs()
print("veri klasörü:", data_root, "| model kayıt klasörü:", models_root)

years = [2017] if HIZLI_DENEME else cfg_data["years"]
splits = train.load_splits(data_root)
local_dir = Path("/content/data_local") if IN_COLAB else None
data, variables = train.load_data(paths.processed_dir(), splits, years, which=("train", "val"), local_dir=local_dir)
y_val = data["val"]["y"]

print(f"yıllar: {years} | kanallar: {variables} | girdi: {data['train']['X'].shape[1:]}")
print(f"eğitim: {len(data['train']['y']):,} | doğrulama: {len(y_val):,}")

veri klasörü: /content/drive/MyDrive/tornado-cnn/data | model kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models
yıllar: [2015, 2017, 2019] | kanallar: ['DBZ', 'VEL', 'WIDTH'] | girdi: (3, 120, 240)
eğitim: 48,703 | doğrulama: 8,040


## Ortak eğitim fonksiyonu

Her model hücresi ayarını `configs/experiments.json` dosyasından okuyan `run_from_config(name)` fonksiyonunu çağırır:
1. Hedef klasörde `results.json` zaten varsa modeli yeniden eğitmeden atlar (`force=True` verilmedikçe).
2. Seçilen `seed` değerini yeniden kurar (aynı batch karıştırması = adil karşılaştırma).
3. `cnn.build_model(arch, ...)` ile modeli kurar ve `train.fit` ile eğitir.
4. Doğrulama kümesinde F1'i en yüksek yapan eşiği seçip `results.json` kaydeder. *(Not: Eşik aynı `val` kümesinde seçildiği için buradaki F1/Precision/Recall değerleri hafifçe iyimserdir; asıl tarafsız ölçüm test aşamasında bu eşik sabitlenerek yapılır.)*

In [4]:
def run_experiment(name: str, arch: str, model_kwargs=None, train_overrides=None,
                   arch_label=None, note="", force=False, short_label=None):
    out_name = f"{name}_hizli" if HIZLI_DENEME else name
    out_dir = paths.models_dir(out_name)
    if not force and (out_dir / "results.json").exists():
        print(f"ATLANDI: {out_dir / 'results.json'} zaten var (yeniden eğitmek için force=True verin).")
        return

    exp_cfg = {**cfg, **(train_overrides or {})}
    unknown = set(exp_cfg) - (set(cfg) | {"scheduler"})
    if unknown:
        raise ValueError(f"train.json içinde olmayan bilinmeyen ayarlar: {unknown}")

    train.set_seed(exp_cfg["seed"])
    w_train, pos_weight = train.class_weights(data["train"]["meta"], exp_cfg["weighting"])
    train_loader, val_loader = train.make_loaders(data, exp_cfg, w_train)

    net = cnn.build_model(
        arch, in_channels=len(variables), dropout=exp_cfg["dropout"], **(model_kwargs or {})
    ).to(device)
    n_params = cnn.count_params(net)

    print(f"=== {arch_label or name} | parametre: {n_params:,} | lr: {exp_cfg['lr']} | "
          f"weighting: {exp_cfg['weighting']} | seed: {exp_cfg['seed']} ===")
    print("kayıt klasörü:", out_dir)

    history = train.fit(net, train_loader, val_loader, exp_cfg, variables, out_dir, pos_weight, device)
    p_val, val_loss = train.predict(net, val_loader, variables, device, pos_weight)
    threshold = train.choose_threshold(y_val, p_val)
    val_metrics = train.metrics(y_val, p_val, threshold)
    best_epoch = train.best_epoch(history, exp_cfg)

    train.save_results(
        out_dir,
        model_name=arch_label or name,
        arch=arch,
        model_kwargs=model_kwargs or {},
        note=note,
        params=n_params,
        config=exp_cfg,
        years=years,
        variables=variables,
        pos_weight=pos_weight,
        best_epoch=best_epoch,
        seed=exp_cfg["seed"],
        short_label=short_label,
        threshold=threshold,
        val_loss=float(val_loss),
        validation=val_metrics,
    )
    print(f"Tamamlandı → en iyi tur: {best_epoch} | PR-AUC: {val_metrics['PR-AUC']:.4f} | "
          f"ROC-AUC: {val_metrics['ROC-AUC']:.4f} | F1 (val eşikli): {val_metrics['F1']:.4f}\n")

    del net
    if device.type == "cuda":
        torch.cuda.empty_cache()


def run_from_config(name: str, seed=None, force=False):
    """Bir deneyi tek seed ile eğitir. Varsayılan seed (train.json) klasör adını değiştirmez;
    diğer seed'ler <ad>_s<seed> klasörüne yazılır (karar M3), böylece birbirinin üstüne yazmaz."""
    spec = EXPERIMENTS[name]
    seed = cfg["seed"] if seed is None else int(seed)
    overrides = {**spec.get("train", {}), "seed": seed}
    extra = "" if seed == cfg["seed"] else f" · seed {seed}"
    return run_experiment(
        name=train.seed_folder(spec["name"], seed, cfg["seed"]),
        arch=spec["arch"],
        model_kwargs=spec.get("model_kwargs", {}),
        train_overrides=overrides,
        arch_label=spec.get("label", spec["name"]) + extra,
        short_label=(spec["short"] + extra) if spec.get("short") else None,
        note=spec.get("note", ""),
        force=force,
    )


def run_seeds(name: str, force=False):
    """experiments.json'daki "seeds" listesinin hepsini eğitir (bitmiş olanlar atlanır)."""
    for s in EXPERIMENTS[name].get("seeds", [cfg["seed"]]):
        run_from_config(name, seed=s, force=force)

## 1. TornadoCNN — Global Average Pooling (`05_cnn_avg`)

Zorunlu ablasyon: `03_baseline` ile birebir aynı ayarlar, yalnızca `AdaptiveMaxPool2d(1)` yerine `AdaptiveAvgPool2d(1)`.

In [5]:
run_from_config("05_cnn_avg")

=== TornadoCNN (Global Avg) | parametre: 105,761 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_avg
tur  1 | eğitim kaybı 1.1440 | doğrulama kaybı 1.1584 | AUC 0.7611 | PR-AUC 0.2144 | 29 sn  ★ en iyi
tur  2 | eğitim kaybı 1.0966 | doğrulama kaybı 1.1160 | AUC 0.7803 | PR-AUC 0.2262 | 28 sn  ★ en iyi
tur  3 | eğitim kaybı 1.0034 | doğrulama kaybı 0.9774 | AUC 0.8558 | PR-AUC 0.4114 | 28 sn  ★ en iyi
tur  4 | eğitim kaybı 0.8813 | doğrulama kaybı 0.9838 | AUC 0.8486 | PR-AUC 0.4370 | 29 sn  ★ en iyi
tur  5 | eğitim kaybı 0.8341 | doğrulama kaybı 0.9034 | AUC 0.8660 | PR-AUC 0.4822 | 29 sn  ★ en iyi
tur  6 | eğitim kaybı 0.7958 | doğrulama kaybı 0.8341 | AUC 0.8824 | PR-AUC 0.5410 | 29 sn  ★ en iyi
tur  7 | eğitim kaybı 0.7795 | doğrulama kaybı 0.8363 | AUC 0.8822 | PR-AUC 0.5552 | 29 sn  ★ en iyi
tur  8 | eğitim kaybı 0.7589 | doğrulama kaybı 0.8868 | AUC 0.8781 | PR-AUC 0.5629 | 29 sn  ★ en iyi
tur  9 | eğitim kaybı 0.

## 2. ResNet-18 — Ön-eğitimli (`05_resnet18_pretrained`)

Zorunlu ikinci mimari: ImageNet ön-eğitimli ResNet-18 (`lr=1e-4`, `[-1, 1]` radar ölçeği).

In [6]:
run_from_config("05_resnet18_pretrained")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


=== ResNet-18 (ön-eğitimli) | parametre: 11,177,025 | lr: 0.0001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_resnet18_pretrained
tur  1 | eğitim kaybı 0.8275 | doğrulama kaybı 0.8482 | AUC 0.8811 | PR-AUC 0.5029 | 95 sn  ★ en iyi
tur  2 | eğitim kaybı 0.5075 | doğrulama kaybı 1.0069 | AUC 0.8709 | PR-AUC 0.5380 | 95 sn  ★ en iyi
tur  3 | eğitim kaybı 0.2796 | doğrulama kaybı 1.6887 | AUC 0.8391 | PR-AUC 0.4429 | 95 sn
tur  4 | eğitim kaybı 0.2002 | doğrulama kaybı 1.4214 | AUC 0.8724 | PR-AUC 0.4710 | 95 sn
tur  5 | eğitim kaybı 0.1239 | doğrulama kaybı 2.2080 | AUC 0.8490 | PR-AUC 0.4064 | 95 sn
tur  6 | eğitim kaybı 0.1069 | doğrulama kaybı 2.5574 | AUC 0.8227 | PR-AUC 0.4095 | 96 sn  → erken durdurma (en iyi tur: 2)
En iyi tur: 2 (doğrulama PR-AUC 0.5380). Bu turun ağırlıkları yüklendi.
Tamamlandı → en iyi tur: 2 | PR-AUC: 0.5380 | ROC-AUC: 0.8709 | F1 (val eşikli): 0.5491



## 3. ResNet-18 — Sıfırdan (`05_resnet18_scratch`)

Zorunlu karşılaştırma: Ön-eğitimin katkısını adil ölçmek için `05_resnet18_pretrained` ile aynı `lr=1e-4`.

In [7]:
run_from_config("05_resnet18_scratch")

=== ResNet-18 (sıfırdan) | parametre: 11,177,025 | lr: 0.0001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_resnet18_scratch
tur  1 | eğitim kaybı 0.9985 | doğrulama kaybı 1.0991 | AUC 0.8278 | PR-AUC 0.3606 | 96 sn  ★ en iyi
tur  2 | eğitim kaybı 0.7783 | doğrulama kaybı 0.8906 | AUC 0.8750 | PR-AUC 0.4915 | 96 sn  ★ en iyi
tur  3 | eğitim kaybı 0.6341 | doğrulama kaybı 1.2390 | AUC 0.8609 | PR-AUC 0.4504 | 95 sn
tur  4 | eğitim kaybı 0.4665 | doğrulama kaybı 1.0900 | AUC 0.8470 | PR-AUC 0.4239 | 95 sn
tur  5 | eğitim kaybı 0.3298 | doğrulama kaybı 1.3688 | AUC 0.8383 | PR-AUC 0.3560 | 95 sn
tur  6 | eğitim kaybı 0.2354 | doğrulama kaybı 1.3918 | AUC 0.8635 | PR-AUC 0.4731 | 95 sn  → erken durdurma (en iyi tur: 2)
En iyi tur: 2 (doğrulama PR-AUC 0.4915). Bu turun ağırlıkları yüklendi.
Tamamlandı → en iyi tur: 2 | PR-AUC: 0.4915 | ROC-AUC: 0.8750 | F1 (val eşikli): 0.5275



## 4. TornadoCNN + BatchNorm (`05_cnn_bn`, isteğe bağlı)

Evrişim bloklarına `BatchNorm2d` eklemenin yakınsama hızı ve doğrulama kaybı üzerindeki etkisini ölçer.

In [8]:
run_from_config("05_cnn_bn")

=== TornadoCNN (Global Max + BN) | parametre: 106,241 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_bn
tur  1 | eğitim kaybı 0.9812 | doğrulama kaybı 0.8942 | AUC 0.8733 | PR-AUC 0.4878 | 37 sn  ★ en iyi
tur  2 | eğitim kaybı 0.8210 | doğrulama kaybı 0.7926 | AUC 0.8936 | PR-AUC 0.5699 | 37 sn  ★ en iyi
tur  3 | eğitim kaybı 0.7766 | doğrulama kaybı 0.9179 | AUC 0.8840 | PR-AUC 0.5568 | 37 sn
tur  4 | eğitim kaybı 0.7380 | doğrulama kaybı 0.8445 | AUC 0.8837 | PR-AUC 0.5672 | 37 sn
tur  5 | eğitim kaybı 0.7145 | doğrulama kaybı 0.8223 | AUC 0.8918 | PR-AUC 0.5744 | 38 sn  ★ en iyi
tur  6 | eğitim kaybı 0.6708 | doğrulama kaybı 0.8180 | AUC 0.8923 | PR-AUC 0.5922 | 37 sn  ★ en iyi
tur  7 | eğitim kaybı 0.6466 | doğrulama kaybı 0.8718 | AUC 0.8845 | PR-AUC 0.5761 | 37 sn
tur  8 | eğitim kaybı 0.6200 | doğrulama kaybı 1.0326 | AUC 0.8822 | PR-AUC 0.5578 | 37 sn
tur  9 | eğitim kaybı 0.5848 | doğrulama kaybı 0.8581 | AUC 

## 5. TornadoCNN + CoordConv (`05_cnn_coord`, isteğe bağlı)

Karar K4: Kutupsal ızgarada bir pikselin fiziksel boyutu ve ışının yüksekliği radara uzaklıkla değişir; evrişim bunu kendisi bilemez. Bu deneyde girdiye radara uzaklık `r` ve `1/r` kanalları eklenir (3 radar kanalı + 2 = 5 kanal). Kanallar her örneğin `rng_min` değerinden modelin içinde üretilir (karar P2): `r` [0, 250 km] → [-1, 1], `1/r` = 10 km / max(r, 10 km) → [-1, 1].

`splits.csv` içinde `rng_min` sütunu olmalı: 02 defterinin 3. bölümü güncel kodla yeniden çalıştırılmış olmalı.

In [9]:
run_from_config("05_cnn_coord")

=== TornadoCNN (Global Max + CoordConv) | parametre: 106,049 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_coord
tur  1 | eğitim kaybı 0.9893 | doğrulama kaybı 0.9125 | AUC 0.8669 | PR-AUC 0.4894 | 31 sn  ★ en iyi
tur  2 | eğitim kaybı 0.8272 | doğrulama kaybı 0.8424 | AUC 0.8792 | PR-AUC 0.5440 | 31 sn  ★ en iyi
tur  3 | eğitim kaybı 0.7844 | doğrulama kaybı 0.8558 | AUC 0.8857 | PR-AUC 0.5573 | 31 sn  ★ en iyi
tur  4 | eğitim kaybı 0.7645 | doğrulama kaybı 0.8513 | AUC 0.8807 | PR-AUC 0.5451 | 31 sn
tur  5 | eğitim kaybı 0.7316 | doğrulama kaybı 0.8033 | AUC 0.8926 | PR-AUC 0.5761 | 31 sn  ★ en iyi
tur  6 | eğitim kaybı 0.6973 | doğrulama kaybı 0.8356 | AUC 0.8902 | PR-AUC 0.5516 | 31 sn
tur  7 | eğitim kaybı 0.6699 | doğrulama kaybı 0.8144 | AUC 0.8976 | PR-AUC 0.6022 | 31 sn  ★ en iyi
tur  8 | eğitim kaybı 0.6460 | doğrulama kaybı 0.9329 | AUC 0.8765 | PR-AUC 0.5351 | 31 sn
tur  9 | eğitim kaybı 0.6246 | doğrulama

## 6. TornadoCNN + augmentation (`05_cnn_aug`, isteğe bağlı)

Karar A2: Temel modelle aynı ayarlar, yalnızca eğitimde fiziksel olarak geçerli bozulmalar (`"augment": true`):

* **Azimut kaydırması** ±4 hücre (±2°) ve **mesafe kaydırması** ±5 hücre (±1,25 km). Boşalan kenar "ölçüm yok" ile dolar; `roll` kullanılmaz.
* **Gauss gürültüsü** yalnızca ölçülmüş piksellere: DBZ σ = 1 dBZ, VEL σ = 1 m/s, WIDTH σ = 0,25 m/s.
* **Her iki sınıfa** uygulanır: model "bozulma izi = hortum" kısayolunu öğrenemez ve sınıf oranı (dolayısıyla `pos_weight`) değişmez.
* Doğrulama ve test kümelerine dokunulmaz. Ayna ve döndürme yoktur (karar A3: radyal hız ve kutupsal ızgara).
* CoordConv ile birlikte kullanılırsa mesafe kaydırması `r` kanalına da yansıtılır.

In [10]:
run_from_config("05_cnn_aug")

=== TornadoCNN (Global Max + augmentation) | parametre: 105,761 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_aug
tur  1 | eğitim kaybı 1.1336 | doğrulama kaybı 1.0018 | AUC 0.8518 | PR-AUC 0.4249 | 30 sn  ★ en iyi
tur  2 | eğitim kaybı 0.9449 | doğrulama kaybı 0.8713 | AUC 0.8736 | PR-AUC 0.5293 | 31 sn  ★ en iyi
tur  3 | eğitim kaybı 0.8403 | doğrulama kaybı 0.8638 | AUC 0.8782 | PR-AUC 0.5253 | 31 sn
tur  4 | eğitim kaybı 0.7991 | doğrulama kaybı 0.9206 | AUC 0.8628 | PR-AUC 0.5022 | 30 sn
tur  5 | eğitim kaybı 0.7717 | doğrulama kaybı 0.8610 | AUC 0.8799 | PR-AUC 0.5280 | 30 sn
tur  6 | eğitim kaybı 0.7383 | doğrulama kaybı 0.8511 | AUC 0.8813 | PR-AUC 0.5776 | 30 sn  ★ en iyi
tur  7 | eğitim kaybı 0.7183 | doğrulama kaybı 0.8920 | AUC 0.8781 | PR-AUC 0.5707 | 30 sn
tur  8 | eğitim kaybı 0.7049 | doğrulama kaybı 0.9083 | AUC 0.8731 | PR-AUC 0.5293 | 31 sn
tur  9 | eğitim kaybı 0.6925 | doğrulama kaybı 0.8580 | AUC

## 7. TornadoCNN + label smoothing (`05_cnn_ls`, isteğe bağlı)

Karar O2: Temel modelle aynı ayarlar, yalnızca eğitim kaybında **pozitif etiketler 1 → 0,9** (`"label_smoothing": 0.1`); negatifler 0 kalır. Amaç, kısa ömürlü ya da radarın zor gördüğü hortumlardaki etiket gürültüsüne karşı modeli aşırı emin olmaktan alıkoymak (TorNet de label smoothing kullanıyor).

**Neden simetrik değil?** Simetrik smoothing'de (0 → 0,05) her negatifin küçük pozitif hedefi de `pos_weight ≈ 11,7` ile çarpılır: 62 926 × 0,05 × 11,7 ≈ 36 800, pozitiflerin 5 388 × 0,95 × 11,7 ≈ 59 900'üne karşı. Yani negatifler pozitif yöndeki kaybın ~%38'ini oluşturur ve modeli her şeye "biraz hortum" demeye iter.

**Sınıf dengesi örnek ağırlığıyla:** `BCEWithLogitsLoss(pos_weight)` pozitif örnekte yalnızca "hortum" terimini 11,7 ile çarpar; 0,9 hedefinde kaybı en küçük yapan çıktıık 0,991 olurdu ve smoothing neredeyse etkisiz kalırdı. Bu yüzden smoothing açıkken pozitif örneğin kaybının tamamı 11,7 ile çarpılır (`train.train_loss`). Smoothing kapalıyken iki yol matematiksel olarak aynıdır; diğer modeller etkilenmez.

Doğrulama kaybı ve bütün ölçütler gerçek etiketlerle hesaplanır.

In [11]:
run_from_config("05_cnn_ls")

=== TornadoCNN (Global Max + label smoothing) | parametre: 105,761 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_ls
tur  1 | eğitim kaybı 1.1467 | doğrulama kaybı 1.0298 | AUC 0.8272 | PR-AUC 0.3843 | 28 sn  ★ en iyi
tur  2 | eğitim kaybı 0.9776 | doğrulama kaybı 0.8973 | AUC 0.8797 | PR-AUC 0.5316 | 28 sn  ★ en iyi
tur  3 | eğitim kaybı 0.9157 | doğrulama kaybı 0.9888 | AUC 0.8764 | PR-AUC 0.5170 | 28 sn
tur  4 | eğitim kaybı 0.8875 | doğrulama kaybı 0.9316 | AUC 0.8664 | PR-AUC 0.5286 | 28 sn
tur  5 | eğitim kaybı 0.8651 | doğrulama kaybı 0.9280 | AUC 0.8692 | PR-AUC 0.5275 | 28 sn
tur  6 | eğitim kaybı 0.8396 | doğrulama kaybı 0.9412 | AUC 0.8732 | PR-AUC 0.5476 | 28 sn  ★ en iyi
tur  7 | eğitim kaybı 0.8183 | doğrulama kaybı 0.8733 | AUC 0.8847 | PR-AUC 0.5879 | 28 sn  ★ en iyi
tur  8 | eğitim kaybı 0.7984 | doğrulama kaybı 1.0243 | AUC 0.8710 | PR-AUC 0.5412 | 28 sn
tur  9 | eğitim kaybı 0.7711 | doğrulama kaybı 

## 8. TornadoCNN · mit ağırlıkları (`03_mit`)

Karar S3: Temel modelle aynı mimari ve ayarlar, yalnızca sınıf ağırlığı farklı: TorNet'in kategori ağırlıkları (WRN 0,5; NUL 1; EF0–1 1; EF2+ 2) ve `pos_weight = 1`. Bu şemada sınıf dengesizliği büyük ölçüde dengelenmez; model daha düşük skorlar üretir ve eşik de buna göre düşer. PR-AUC ve ROC-AUC eşikten bağımsız olduğu için karşılaştırma adildir. TorNet ağırlıkların getirisini "marjinal" bulmuş; bizde de öyle çıkarsa bu da bir sonuçtur.

In [12]:
run_from_config("03_mit")

=== TornadoCNN (Global Max, mit) | parametre: 105,761 | lr: 0.001 | weighting: mit | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/03_mit
tur  1 | eğitim kaybı 0.2595 | doğrulama kaybı 0.2567 | AUC 0.8355 | PR-AUC 0.3951 | 29 sn  ★ en iyi
tur  2 | eğitim kaybı 0.1925 | doğrulama kaybı 0.2068 | AUC 0.8818 | PR-AUC 0.5537 | 29 sn  ★ en iyi
tur  3 | eğitim kaybı 0.1717 | doğrulama kaybı 0.1932 | AUC 0.8827 | PR-AUC 0.5625 | 29 sn  ★ en iyi
tur  4 | eğitim kaybı 0.1611 | doğrulama kaybı 0.1967 | AUC 0.8666 | PR-AUC 0.5652 | 29 sn  ★ en iyi
tur  5 | eğitim kaybı 0.1525 | doğrulama kaybı 0.2000 | AUC 0.8851 | PR-AUC 0.5725 | 29 sn  ★ en iyi
tur  6 | eğitim kaybı 0.1417 | doğrulama kaybı 0.1961 | AUC 0.8796 | PR-AUC 0.5705 | 29 sn
tur  7 | eğitim kaybı 0.1367 | doğrulama kaybı 0.1910 | AUC 0.8803 | PR-AUC 0.5883 | 29 sn  ★ en iyi
tur  8 | eğitim kaybı 0.1294 | doğrulama kaybı 0.2138 | AUC 0.8664 | PR-AUC 0.5449 | 29 sn
tur  9 | eğitim kaybı 0.1223 | doğrulama kaybı 0.2

## 9. Ana karşılaştırma: 3 seed (karar M3)

Zorunlu karşılaştırmanın iki modeli **TornadoCNN (temel)** ve **ResNet-18 (ön-eğitimli)**, `experiments.json`'daki `"seeds": [42, 7, 123]` ile üçer kez eğitilir. Sonuç 06 defterinde ortalama ± std olarak verilir. Temel modelin seed std'si ablasyonlar için "gürültü ölçüsü"dür: bu std'nin altındaki farklar yorumlanmaz. Ablasyonlar tek seed (42) ile kalır.

* Seed 42 bugünkü klasör adını kullanır: `03_baseline` (03 defteri) ve `05_resnet18_pretrained` (yukarıdaki 2. bölüm). Bunlar eğitilmişse atlanır.
* Diğer seed'ler ayrı klasörlere yazılır: `03_baseline_s7`, `03_baseline_s123`, `05_resnet18_pretrained_s7`, `05_resnet18_pretrained_s123`.
* Yeni eğitim sayısı: 4 (2 model × 2 ek seed).

In [13]:
run_seeds("03_baseline")               # seed 42 = 03 defterinin modeli; 7 ve 123 burada
run_seeds("05_resnet18_pretrained")

ATLANDI: /content/drive/MyDrive/tornado-cnn/models/03_baseline/results.json zaten var (yeniden eğitmek için force=True verin).
=== TornadoCNN (Global Max, pos_weight) · seed 7 | parametre: 105,761 | lr: 0.001 | weighting: pos_weight | seed: 7 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/03_baseline_s7
tur  1 | eğitim kaybı 1.0911 | doğrulama kaybı 0.9384 | AUC 0.8540 | PR-AUC 0.4748 | 29 sn  ★ en iyi
tur  2 | eğitim kaybı 0.8480 | doğrulama kaybı 0.8431 | AUC 0.8796 | PR-AUC 0.5562 | 29 sn  ★ en iyi
tur  3 | eğitim kaybı 0.7910 | doğrulama kaybı 0.8727 | AUC 0.8701 | PR-AUC 0.5408 | 29 sn
tur  4 | eğitim kaybı 0.7498 | doğrulama kaybı 0.8463 | AUC 0.8798 | PR-AUC 0.5671 | 29 sn  ★ en iyi
tur  5 | eğitim kaybı 0.7141 | doğrulama kaybı 1.0394 | AUC 0.8567 | PR-AUC 0.5314 | 29 sn
tur  6 | eğitim kaybı 0.6881 | doğrulama kaybı 0.9580 | AUC 0.8676 | PR-AUC 0.5248 | 29 sn
tur  7 | eğitim kaybı 0.6472 | doğrulama kaybı 0.9041 | AUC 0.8619 | PR-AUC 0.5255 | 29 sn
tur  8 | eğiti

## 10. Toplu koşum (isteğe bağlı)

`configs/experiments.json` dosyasına eklediğin yeni deneyleri tek satırda çalıştırmak için kullanabilirsin (bitmiş deneyler otomatik atlanır).

In [14]:
# Config'deki tüm zorunlu deneyleri tek seferde çalıştırmak için:
# for exp in exp_registry["experiments"]:
#     if exp.get("required", True):
#         run_from_config(exp["name"])

# Çoklu seed: experiments.json'da "seeds" listesi + run_seeds(ad) (9. bölüm).
